# When is migration worth it?

Run a reproducible comparison of native MuJoCo and the GPU workflow on the two-cube stacking task. Start with one world and 16 worlds, inspect task success, then compare the repeated timings. The reports retain the hardware, software, configuration and raw samples.

These are community measurements on the recorded machine, not official product benchmarks. The default experiment replays the same task commands through the same MJCF model. An optional workflow experiment includes the actual host controller and Newton migration; it is reported separately because model reconstruction and contact interfaces differ.

Read the [benchmark guide](../BENCHMARK.md) for timing scope, CPU worker counts and interpretation. This notebook does not run the long cloth-and-cable exercise.

## Setup

**Local or Brev:** open this notebook from the checkout and select the tutorial's Python 3.12 environment. The [setup guide](../README.md#setup) installs the pinned dependencies. An existing Brev instance can use the same commands as a local machine.

**Google Colab:** select a GPU runtime. Running the setup cells downloads the selected public source revision and installs the hash-locked environment into a separate Python 3.12 virtual environment. Benchmark subprocesses use that interpreter, leaving the hosted kernel's physics packages unchanged. Before publishing results, replace `REF` with the full commit printed below and use a fresh checkout directory.

Colab's [GPU allocation and runtime limits vary](https://research.google.com/colaboratory/faq.html). This code path is provided for validation; a T4/L4 or hosted deployment is not claimed as tested. A Colab CPU baseline belongs to that runtime, not your laptop.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

# This public review branch can move. Use its full resolved commit for publication.
REPO_URL = "https://github.com/johnnynunez/accelerated-computing-hub.git"
REF = "feature/newton-task-benchmark"
COLAB_CHECKOUT = Path("/content/newton-benchmark-source")
TUTORIAL_RELATIVE = Path("tutorials/newton")

if IN_COLAB:
    REPO_ROOT = COLAB_CHECKOUT
    git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
    if not REPO_ROOT.exists():
        REPO_ROOT.mkdir(parents=True)
        subprocess.run(["git", "init", str(REPO_ROOT)], check=True, env=git_env)
        subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "add", "origin", REPO_URL],
                       check=True, env=git_env)
        subprocess.run(["git", "-C", str(REPO_ROOT), "sparse-checkout", "init", "--cone"],
                       check=True, env=git_env)
        subprocess.run(["git", "-C", str(REPO_ROOT), "sparse-checkout", "set", str(TUTORIAL_RELATIVE)],
                       check=True, env=git_env)
    actual_remote = subprocess.check_output(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"], text=True).strip()
    if actual_remote != REPO_URL:
        raise RuntimeError("The existing checkout uses another repository; choose a fresh COLAB_CHECKOUT.")
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", REF],
                   check=True, env=git_env)
    requested_commit = subprocess.check_output(
        ["git", "-C", str(REPO_ROOT), "rev-parse", "FETCH_HEAD"], text=True).strip()
    existing = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "--verify", "HEAD"],
                              capture_output=True, text=True)
    if existing.returncode == 0 and existing.stdout.strip() != requested_commit:
        raise RuntimeError("REF changed. Existing files were preserved; choose a fresh COLAB_CHECKOUT.")
    if existing.returncode != 0:
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", requested_commit],
                       check=True, env=git_env)
else:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    REPO_ROOT = next((p for p in candidates if
                      (p / TUTORIAL_RELATIVE / "notebooks/newton/requirements.lock.txt").is_file()), None)
    if REPO_ROOT is None:
        raise RuntimeError("Open the notebook inside the accelerated-computing-hub checkout.")

TUTORIAL_DIR = REPO_ROOT / TUTORIAL_RELATIVE
PART3 = TUTORIAL_DIR / "notebooks/newton/part3"
SCRIPT = PART3 / "migration_benchmark.py"
if not SCRIPT.is_file():
    raise RuntimeError("This source revision does not contain migration_benchmark.py.")
revision = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
                          capture_output=True, text=True)
print("Source:", revision.stdout.strip() if revision.returncode == 0 else
      "Image snapshot without Git metadata; retain the report's source fingerprints.")
print("Tutorial:", TUTORIAL_DIR)


In [ ]:
import hashlib

# Explicit setup for a hosted runtime. Local/Brev users select their prepared kernel.
if IN_COLAB:
    runtime_root = Path("/content/newton-benchmark-runtime")
    tools_env = runtime_root / "tools"
    benchmark_env = runtime_root / "python312"
    uv = tools_env / "bin/uv"
    if not uv.is_file():
        subprocess.run([sys.executable, "-m", "venv", str(tools_env)], check=True)
        subprocess.run([str(tools_env / "bin/python"), "-m", "pip", "install", "uv==0.12.5"],
                       check=True)
    BENCHMARK_PYTHON = benchmark_env / "bin/python"
    if not BENCHMARK_PYTHON.is_file():
        subprocess.run([str(uv), "venv", "--python", "3.12", str(benchmark_env)], check=True)
    subprocess.run([str(uv), "pip", "sync", "--python", str(BENCHMARK_PYTHON),
                    "--require-hashes", str(TUTORIAL_DIR / "notebooks/newton/requirements.lock.txt")],
                   check=True)
else:
    BENCHMARK_PYTHON = Path(sys.executable)

BENCHMARK_ENV = {**os.environ, "PXR_WORK_THREAD_LIMIT": "1"}
subprocess.run([str(BENCHMARK_PYTHON), "-c",
                "import sys; print(sys.executable); print(sys.version); "
                "assert sys.version_info[:2] == (3, 12), 'Use the pinned Python 3.12 environment.'"],
               check=True, env=BENCHMARK_ENV)
lock = TUTORIAL_DIR / "notebooks/newton/requirements.lock.txt"
print("Requirements lock SHA-256:", hashlib.sha256(lock.read_bytes()).hexdigest())


## Select the workload

The default `replay` scope compares CPU serial, CPU worker-pool and GPU batch execution of the same model and commands. It excludes controller computation. Each measured world still has to complete the task; validation and finite-state/capacity checks cannot be replaced by a timing alone. Both backends record full physics state after each substep. GPU download and task validation are timed separately; compare time to checked host output as well as simulation throughput. CPU double precision and GPU single precision need not produce identical trajectories.

`workflow` compares the one-world native MuJoCo and Newton applications with the actual host controller, control transfers and both cubes' pose/velocity observations at 50 Hz. Recording into RAM or VRAM belongs to its simulation timer; final host download and validation are separate costs. It has different model/contact interfaces and is not an identical-model replay. Use `both` to collect the two experiments separately.

Keep the full episode when reducing memory use: reduce `WORLDS`, not task duration. After the small batch passes, `[1, 16, 64, 256]` is an optional workstation sweep. Use at least five repetitions for numbers intended for publication. `CPU_ONLY = True` collects an explicit CPU baseline; it does not provide a GPU speedup.

The common replay configuration sets 100 solver iterations, 50 line-search iterations and `impratio = 100` on both backends. Task acceptance requires a whole-cube lift, at least 5 cm of continuous airborne carry, and a stable stack during the final second. The [benchmark guide](../BENCHMARK.md#what-counts-as-a-successful-task) gives the exact settling thresholds and retained diagnostics.

In [ ]:
from datetime import datetime, timezone
import shlex

ROBOT = "so101"          # or "rebot"
WORLDS = [1, 16]
DEVICE = "cuda:0"        # one selected GPU per invocation
REPEATS = 3
WARMUPS = 1
CPU_THREADS = 4          # CPU worker-pool size; use available cores responsibly
MAX_TRAJECTORY_MIB = 1024
SCOPE = "replay"         # "replay", "workflow", or "both"
CPU_ONLY = False
RUN_BENCHMARK = False    # inspect preflight and configuration, then set True

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
REPORT_DIR = PART3 / ".generated/benchmarks" / f"{ROBOT}-{stamp}"
common = [str(BENCHMARK_PYTHON), str(SCRIPT), "--robot", ROBOT,
          "--worlds", *map(str, WORLDS), "--device", DEVICE,
          "--repeats", str(REPEATS), "--warmups", str(WARMUPS),
          "--cpu-threads", str(CPU_THREADS), "--scope", SCOPE,
          "--max-trajectory-mib", str(MAX_TRAJECTORY_MIB)]
if CPU_ONLY:
    common.append("--cpu-only")
command = [*common, "--output-dir", str(REPORT_DIR)]
print(shlex.join(command))


## Preflight

This inspects the selected environment and devices without producing performance numbers. An absent GPU remains absent; a real GPU measurement must fail rather than silently falling back to CPU. Check the actual GPU and driver, available memory and any concurrent applications before running. Shared-machine observations must be described as such.

In [ ]:
subprocess.run([*common, "--preflight", "--output-dir", str(REPORT_DIR / "preflight")],
               cwd=PART3, env=BENCHMARK_ENV, check=True)


## Measure and inspect

Setup, compilation and warm-up can take longer than a warmed run. The command prints each case and its status; detailed worker output is saved in the corresponding `.log` file in the report directory. A subprocess error must be investigated; the notebook does not substitute a successful-looking summary.

Read task success and scope before comparing throughput. Report the CPU worker count alongside speedup, and preserve individual samples. Compilation/setup cost matters when the real application only runs a short episode.

In [ ]:
if RUN_BENCHMARK:
    subprocess.run(command, cwd=PART3, env=BENCHMARK_ENV, check=True)
else:
    print("Preflight only. Set RUN_BENCHMARK = True to collect measured results.")


In [ ]:
from IPython.display import Markdown, display

summary = REPORT_DIR / "summary.md"
if (REPORT_DIR / "results.json").is_file() and summary.is_file():
    display(Markdown(summary.read_text(encoding="utf-8")))
    print("Raw reports:", REPORT_DIR)
else:
    print("No measured result from this invocation. Preflight is not a benchmark result.")


## Save the evidence

The archive below includes JSON, CSV, Markdown and worker logs from this invocation, excluding compiled caches. On Colab, set `DOWNLOAD_RESULTS = True` to download it before the runtime disconnects. Review the reports before sharing them.

For the article, retain the exact source revision, CPU/GPU/RAM, driver and package versions, world counts, CPU worker count, repetitions, timing scope, task outcomes and machine load. Ask another developer to reproduce the result on the selected accessible GPU. This notebook does not establish support or performance on untested hardware, Brev or Colab.

In [ ]:
import zipfile

DOWNLOAD_RESULTS = False
if (REPORT_DIR / "results.json").is_file():
    archive = REPORT_DIR.with_suffix(".zip")
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for item in sorted(REPORT_DIR.rglob("*")):
            relative = item.relative_to(REPORT_DIR)
            if (item.is_file() and item.suffix in {".json", ".csv", ".md", ".log"}
                    and ".warp-cache" not in relative.parts):
                bundle.write(item, relative)
    print("Results archive:", archive)
    if IN_COLAB and DOWNLOAD_RESULTS:
        from google.colab import files
        files.download(str(archive))
else:
    print("Run the benchmark before exporting measured results.")


## Continue

Repeat with the other robot or another GPU in a separate output directory. Keep the same source commit, configuration and acceptance checks when comparing hardware. See the [benchmark guide](../BENCHMARK.md) for CPU/GPU interpretation and the [migration lesson](02__mujoco_to_newton.ipynb) for the code changes behind the workflow experiment.